# 07｜集成學習與隨機森林

從上到下執行程式格。第一格安裝缺少的套件，第二格準備本章所需設定。資料在首次使用時自動下載。

## 投影片與程式的閱讀方式

對應 `slides/07_集成學習與隨機森林.md`，目前 48 頁。逐圖完整表見 `programs/SLIDE_NOTEBOOK_MAP_05_25.md`。

大數法則、bootstrap/OOB、AdaBoost 重加權、boosting 曲線、OOF 評估範圍。

保留教材原圖；同概念重畫不保證原書分數相同。架構圖以機制或形狀對照，完整模型案例的範圍在各格說明。

| 程式代碼 | 投影片頁次 | 本格學習內容 |
|---|---|---|
| 07-03 | 4、5、6、8、9、41 | 1. 硬投票 vs 軟投票，以及投影片手算 |
| 07-04 | 10、11、12、13、14、42 | 2. Bagging：不同抽樣資料訓練同一種模型 |
| 07-05 | 4、5、7、10、12、14、48 | 投票多樣性與 Bootstrap 成員圖 |
| 07-06 | 15、16、17、43 | 3. 隨機森林、Extra Trees 與特徵重要度 |
| 07-07 | 20、22、23、24、25 | 4. 手算 AdaBoost 第一輪 |
| 07-08 | 20、21、22、23、25、48 | AdaBoost 逐輪加權與二種 alpha 寫法 |
| 07-09 | 3、19、26、27、28、29、30、31、32、44 | 5. Gradient Boosting：平方損失時逐棵學殘差 |
| 07-10 | 36、37、38、39、40、45 | 6. Stacking 與洩漏：第二層必須用折外預測 |
| 07-11 | 34、35、45、46、47 | 7. 同一切分：單樹、森林、提升樹 |
| 07-12 | 31、32、34、35、36、37、38、40、48 | Boosting 的輪數、驗證與分箱 |

In [ ]:
# 第 1 格：安裝本 Notebook 實際使用的套件。
# 程式 07-01｜投影片 07：2 頁
import importlib.util, subprocess, sys
_needed = {'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.0,<3', 'matplotlib': 'matplotlib>=3.7,<4', 'sklearn': 'scikit-learn>=1.4,<2'}
_missing = [spec for module, spec in _needed.items() if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_missing])

In [ ]:
# 第 2 格：本 Notebook 的輸出位置與繪圖設定。
# 程式 07-02｜投影片 07：2、17 頁
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
ROOT = Path.cwd()
if ROOT.name == "notebooks" and (ROOT.parent / "pyproject.toml").exists():
    ROOT = ROOT.parent
OUT = ROOT / ("outputs" if (ROOT / "pyproject.toml").exists() else "mlcourse_outputs")
plt.rcParams.update({"figure.figsize": (8, 4.5), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.2, "font.size": 11})

def savefig(name):
    folder = OUT / "figures"
    folder.mkdir(parents=True, exist_ok=True)
    plt.gcf().savefig(folder / f"{name}.png", dpi=150, bbox_inches="tight")
    svg_path = folder / f"{name}.svg"
    plt.gcf().savefig(svg_path, bbox_inches="tight", metadata={"Date": None})
    svg_path.write_text("\n".join(line.rstrip() for line in svg_path.read_text().splitlines()) + "\n")
    plt.show()
    plt.close(plt.gcf())

def report(name, values):
    folder = OUT / "reports"
    folder.mkdir(parents=True, exist_ok=True)
    (folder / f"{name}.json").write_text(
        json.dumps(values, ensure_ascii=False, indent=2, default=float) + "\n")

DATA = ROOT / ("data" if (ROOT / "pyproject.toml").exists() else "mlcourse_data")
DATA.mkdir(parents=True, exist_ok=True)

# MNIST 只供分類示範使用，首次執行時由 OpenML 下載。
def data_path(name):
    if name != "mnist_784_v1.npz":
        raise ValueError(f"此 Notebook 不使用 {name}")
    path = DATA / name
    if not path.exists():
        from sklearn.datasets import fetch_openml
        ds = fetch_openml("mnist_784", version=1, as_frame=False,
                          data_home=DATA / "openml_cache", parser="auto")
        X, y = ds.data.astype(np.uint8), ds.target.astype(np.uint8)
        assert X.shape == (70000, 784) and y.shape == (70000,)
        np.savez_compressed(path, X=X, y=y)
    with np.load(path) as archive:
        assert archive["X"].shape == (70000, 784)
        assert archive["y"].shape == (70000,)
    return path

## 1. 硬投票 vs 軟投票，以及投影片手算

硬投票數類別票數；軟投票平均機率再取 argmax。對應投影片活動：三個模型正類機率 `[0.51, 0.51, 0.01]`。

In [ ]:
# 程式 07-03｜投影片 07：4、5、6、8、9、41 頁
probs = np.array([0.51, 0.51, 0.01])
hard = int(np.sum(probs >= 0.5) > len(probs) / 2)      # 兩票正類 -> 1
soft = int(probs.mean() >= 0.5)                        # 平均約 0.343 -> 0
assert hard == 1 and soft == 0 and np.isclose(probs.mean(), 0.343333, atol=1e-5)
print('hard vote:', hard, ' soft vote:', soft, ' mean prob:', round(probs.mean(), 4))

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import VotingClassifier
Xm, ym = make_moons(n_samples=500, noise=.3, random_state=SEED)
Xtr, Xte, ytr, yte = train_test_split(Xm, ym, test_size=.3, random_state=SEED)
base = [('lr', make_pipeline(StandardScaler(), LogisticRegression())),
        ('svc', make_pipeline(StandardScaler(), SVC(probability=True, random_state=SEED))),
        ('tree', DecisionTreeClassifier(max_depth=4, random_state=SEED))]
rows = []
for voting in ['hard', 'soft']:
    vc = VotingClassifier(base, voting=voting).fit(Xtr, ytr)
    rows.append({'voting': voting, 'test_accuracy': vc.score(Xte, yte)})
for name, est in base:
    est.fit(Xtr, ytr); rows.append({'voting': name, 'test_accuracy': est.score(Xte, yte)})
pd.DataFrame(rows).set_index('voting')

## 2. Bagging：不同抽樣資料訓練同一種模型

每個估計器看 bootstrap 抽樣的資料，邊界通常比單棵樹平滑。袋外（OOB）樣本可做不用另留固定驗證集的估計；反覆依 OOB 調參仍有選擇偏差。

In [ ]:
# 程式 07-04｜投影片 07：10、11、12、13、14、42 頁
from sklearn.ensemble import BaggingClassifier
xx, yy = np.meshgrid(np.linspace(Xm[:, 0].min() - .5, Xm[:, 0].max() + .5, 300),
                     np.linspace(Xm[:, 1].min() - .5, Xm[:, 1].max() + .5, 300))
single = DecisionTreeClassifier(random_state=SEED).fit(Xtr, ytr)
bag = BaggingClassifier(DecisionTreeClassifier(random_state=SEED), n_estimators=200,
                        oob_score=True, random_state=SEED, n_jobs=2).fit(Xtr, ytr)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, m, name in [(axes[0], single, 'single tree'), (axes[1], bag, 'bagging x200')]:
    zz = m.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, zz, alpha=.3, cmap='coolwarm')
    ax.scatter(Xtr[:, 0], Xtr[:, 1], c=ytr, cmap='coolwarm', s=12, edgecolor='k', linewidth=.3)
    ax.set_title(f'{name}, test acc={m.score(Xte, yte):.3f}')
savefig('12_bagging_boundary')
print('bagging OOB score:', round(bag.oob_score_, 3), ' | test score:', round(bag.score(Xte, yte), 3))

## 投票多樣性與 Bootstrap 成員圖

大數法則的投幣示意使用獨立 Bernoulli 抽樣；完全同步的模型不會增加投票資訊。抽樣矩陣的0代表該棵樹沒看過該樣本，可用於OOB。

In [ ]:
# 程式 07-05｜投影片 07：4、5、7、10、12、14、48 頁
rng_vote=np.random.default_rng(SEED);coins=rng_vote.binomial(1,.51,(5,4000))
plt.plot(np.arange(1,4001),np.cumsum(coins,axis=1).T/np.arange(1,4001)[:,None],alpha=.7)
plt.axhline(.51,color='k',ls='--');plt.xscale('log');plt.xlabel('Independent draws');plt.ylabel('Running head proportion');savefig('c07_large_numbers')
rng_boot=np.random.default_rng(SEED);counts=np.zeros((8,30),int)
for row in range(8):counts[row]=np.bincount(rng_boot.integers(0,30,30),minlength=30)
assert np.all(counts.sum(axis=1)==30)
plt.imshow(counts,aspect='auto',cmap='Blues');plt.colorbar(label='Times sampled');plt.xlabel('Sample ID');plt.ylabel('Estimator');savefig('c07_bootstrap_membership')
print('Observed OOB fraction:',(counts==0).mean(),'expected:',(1-1/30)**30)
base_votes=np.array([est.predict(Xte[:10]) for _,est in base]);print('Base predictions shape:',base_votes.shape)
print('Hard vote for first 10:',(base_votes.mean(axis=0)>.5).astype(int))

## 3. 隨機森林、Extra Trees 與特徵重要度

森林在每個分割點只考慮隨機特徵子集。用 MNIST 子樣本訓練，把不純度重要度 reshape 回 28×28。重要度可偏好高基數特徵，不能解讀成因果。

In [ ]:
# 程式 07-06｜投影片 07：15、16、17、43 頁
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
with np.load(data_path('mnist_784_v1.npz')) as arch:
    images, labels = arch['X'], arch['y']
idx = np.random.default_rng(SEED).choice(60000, 8000, replace=False)
Xd = (images[idx].astype(np.float32) / 255); yd = labels[idx]
Xdtr, Xdte, ydtr, ydte = train_test_split(Xd, yd, test_size=2000, stratify=yd, random_state=SEED)
rf = RandomForestClassifier(n_estimators=120, max_features='sqrt', n_jobs=2, random_state=SEED).fit(Xdtr, ydtr)
et = ExtraTreesClassifier(n_estimators=120, max_features='sqrt', n_jobs=2, random_state=SEED).fit(Xdtr, ydtr)
fig, ax = plt.subplots(figsize=(4, 4))
im = ax.imshow(rf.feature_importances_.reshape(28, 28), cmap='hot')
ax.set_title('Random Forest pixel importance'); ax.axis('off'); fig.colorbar(im, ax=ax, fraction=.046)
savefig('12_forest_importance')
imp = rf.feature_importances_.reshape(28, 28)
assert imp[10:18, 10:18].mean() > imp[:4, :4].mean()          # 中央像素較常參與有效分割
print('held-out accuracy  RandomForest:', round(rf.score(Xdte, ydte), 3),
      ' ExtraTrees:', round(et.score(Xdte, ydte), 3))

## 4. 手算 AdaBoost 第一輪

對應投影片活動。四筆資料初始權重 $1/4$，第一個模型只錯一筆，$\eta=1$。$r$ 是加權錯誤率、$\alpha=\eta\log\frac{1-r}{r}$。

In [ ]:
# 程式 07-07｜投影片 07：20、22、23、24、25 頁
w = np.full(4, 1 / 4)
wrong = np.array([True, False, False, False])
r = w[wrong].sum()
alpha = 1.0 * np.log((1 - r) / r)
w_new = w * np.exp(alpha * wrong)             # 只放大錯分樣本
w_norm = w_new / w_new.sum()
assert np.isclose(r, .25) and np.isclose(alpha, np.log(3))
assert np.isclose(w_norm[wrong][0], .5) and np.allclose(w_norm[~wrong], 1 / 6)
pd.Series({'r': r, 'alpha': alpha, 'weight_wrong_after': w_norm[wrong][0],
           'weight_correct_after': w_norm[~wrong][0]})

## AdaBoost 逐輪加權與二種 alpha 寫法

三輪二元決策樁。錯分才放大 exp(alpha) 與正負標籤版 exp(-alpha_half*y*h) 在正規化後一致；alpha_half 是前者一半。圖上點大小是該輪訓練前的權重。

In [ ]:
# 程式 07-08｜投影片 07：20、21、22、23、25、48 頁
weights=np.full(len(ytr),1/len(ytr));fig,axes=plt.subplots(1,3,figsize=(12,3.5));ada_rows=[]
for k,ax in enumerate(axes):
    stump=DecisionTreeClassifier(max_depth=1,random_state=SEED+k).fit(Xtr,ytr,sample_weight=weights)
    predictions=stump.predict(Xtr);miss=predictions!=ytr;err=np.dot(weights,miss)
    assert 0<err<.5
    alpha_round=np.log((1-err)/err)
    updated=weights*np.exp(alpha_round*miss);updated/=updated.sum()
    sign_y=2*ytr-1;sign_h=2*predictions-1
    symmetric=weights*np.exp(-.5*alpha_round*sign_y*sign_h);symmetric/=symmetric.sum()
    assert np.allclose(updated,symmetric)
    ax.contourf(xx,yy,stump.predict(np.c_[xx.ravel(),yy.ravel()]).reshape(xx.shape),alpha=.2,cmap='coolwarm')
    ax.scatter(Xtr[:,0],Xtr[:,1],c=ytr,s=weights*len(weights)*12,cmap='coolwarm');ax.set_title(f'Round {k+1}, error={err:.3f}')
    ada_rows.append({'round':k+1,'weighted_error':err,'alpha':alpha_round});weights=updated
savefig('c07_adaboost_weights');pd.DataFrame(ada_rows)

## 5. Gradient Boosting：平方損失時逐棵學殘差

每棵新樹擬合目前的殘差，預測是所有樹的加總乘以學習率。學習率小通常需要更多棵樹（圖 6-10）。

In [ ]:
# 程式 07-09｜投影片 07：3、19、26、27、28、29、30、31、32、44 頁
from sklearn.tree import DecisionTreeRegressor
rng = np.random.default_rng(SEED)
xg = np.sort(rng.uniform(-1, 1, 120))[:, None]
yg = xg[:, 0] ** 2 - 0.3 * xg[:, 0] + rng.normal(0, .1, 120)
g = np.linspace(-1, 1, 300)[:, None]
residual = yg.copy(); cumulative = np.zeros(len(g)); lr = 0.5
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for k in range(3):
    stump = DecisionTreeRegressor(max_depth=2, random_state=SEED).fit(xg, residual)
    axes[0, k].scatter(xg, residual, s=10); axes[0, k].plot(g, stump.predict(g), 'r-')
    axes[0, k].set_title(f'round {k + 1}: fit residual')
    cumulative = cumulative + lr * stump.predict(g)
    axes[1, k].scatter(xg, yg, s=10, alpha=.4); axes[1, k].plot(g, cumulative, 'g-')
    axes[1, k].set_title(f'round {k + 1}: cumulative prediction')
    residual = residual - lr * stump.predict(xg)
savefig('12_gradient_boosting_stages')

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import root_mean_squared_error
lr_rows = []
for lr_ in [1.0, 0.1]:
    for n in [5, 30, 120]:
        gb = GradientBoostingRegressor(learning_rate=lr_, n_estimators=n, max_depth=2, random_state=SEED).fit(xg, yg)
        lr_rows.append({'learning_rate': lr_, 'n_estimators': n, 'train_RMSE': root_mean_squared_error(yg, gb.predict(xg))})
lr_table = pd.DataFrame(lr_rows)
assert lr_table.query('learning_rate == 0.1')['train_RMSE'].iloc[0] > lr_table.query('learning_rate == 1.0')['train_RMSE'].iloc[0]
lr_table

## 6. Stacking 與洩漏：第二層必須用折外預測

第二層（blender）的訓練特徵要用樣本外方式產生。直接用第一層對訓練資料的預測，會給高容量基模型「記住答案」的機會。

OOF 特徵可用來訓練 blender；若要交叉驗證整個 stacking，必須在每個外折內重新建立 OOF 特徵。不能先產生一張 OOF 特徵表，再把它當完全獨立資料做普通 CV。

In [ ]:
# 程式 07-10｜投影片 07：36、37、38、39、40、45 頁
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.model_selection import cross_val_predict
estimators = [('tree', DecisionTreeClassifier(max_depth=4, random_state=SEED)),
              ('svc', make_pipeline(StandardScaler(), SVC(probability=True, random_state=SEED)))]
stack = StackingClassifier(estimators, final_estimator=LogisticRegression(), cv=5).fit(Xtr, ytr)

# 洩漏對照：用「記住答案」的滿樹對訓練資料的預測當第二層特徵
leaky_base = DecisionTreeClassifier(random_state=SEED).fit(Xtr, ytr)
leaky_feat_train = leaky_base.predict_proba(Xtr)[:, 1:2]           # in-sample -> 幾乎完美
proper_feat_train = cross_val_predict(DecisionTreeClassifier(random_state=SEED), Xtr, ytr,
                                      cv=5, method='predict_proba')[:, 1:2]
blender_leaky = LogisticRegression().fit(leaky_feat_train, ytr)
blender_proper = LogisticRegression().fit(proper_feat_train, ytr)
leaky_cv = blender_leaky.score(leaky_feat_train, ytr)
proper_fit_score = blender_proper.score(proper_feat_train, ytr)
# 這是blender的擬合分數，不能再對預先生成的OOF特徵做普通CV並稱獨立驗證。
proper_test_score = blender_proper.score(leaky_base.predict_proba(Xte)[:,1:2], yte)
assert leaky_cv > proper_fit_score
print('stacking test accuracy:', round(stack.score(Xte, yte), 3))
print('blender train acc with in-sample feature (leaky):', round(leaky_cv, 3),
      ' | blender fit score with OOF features:', round(proper_fit_score, 3))
print("Held-out score for OOF-trained blender:",proper_test_score)

## 7. 同一切分：單樹、森林、提升樹

固定 moons 訓練資料與 5 折 CV，比較三種模型的 CV 準確率、標準差與訓練時間。

In [ ]:
# 程式 07-11｜投影片 07：34、35、45、46、47 頁
import time
from sklearn.ensemble import HistGradientBoostingClassifier
rows = []
for name, est in {'shallow_tree': DecisionTreeClassifier(max_depth=3, random_state=SEED),
                  'random_forest_200': RandomForestClassifier(n_estimators=200, n_jobs=2, random_state=SEED),
                  'hist_grad_boosting': HistGradientBoostingClassifier(max_iter=200, random_state=SEED)}.items():
    t0 = time.perf_counter()
    cv = cross_val_score(est, Xtr, ytr, cv=5)
    fit_s = time.perf_counter() - t0
    est.fit(Xtr, ytr)
    rows.append({'model': name, 'CV_accuracy': cv.mean(), 'CV_sd': cv.std(ddof=1),
                 'test_accuracy': est.score(Xte, yte), 'cv_seconds': round(fit_s, 3)})
compare = pd.DataFrame(rows).set_index('model')
report('ensembles', {'vote_check': {'hard': 1, 'soft': 0}, 'bagging_oob': float(bag.oob_score_),
       'adaboost_round1': {'r': float(r), 'alpha': float(alpha)},
       'gb_learning_rate_trees': lr_table.to_dict('records'),
       'blender_in_sample_vs_oof_fit': [float(leaky_cv), float(proper_fit_score)],
       'model_comparison': compare.to_dict()})
compare

## Boosting 的輪數、驗證與分箱

固定一份合成訓練／驗證切分，畫 staged_predict 的驗證曲線。選最佳輪數是模型選擇，不能再把該驗證分數當最終 test；HGB 分箱用分位點的小例子理解。

In [ ]:
# 程式 07-12｜投影片 07：31、32、34、35、36、37、38、40、48 頁
bxtr,bxval,bytr,byval=train_test_split(xg,yg,test_size=.3,random_state=SEED)
plt.figure()
for rate in [.1,1.]:
    gb=GradientBoostingRegressor(n_estimators=150,learning_rate=rate,max_depth=2,random_state=SEED).fit(bxtr,bytr)
    curve=[root_mean_squared_error(byval,pred) for pred in gb.staged_predict(bxval)]
    plt.plot(np.arange(1,151),curve,label=f'eta={rate}');print(rate,'best validation stage:',np.argmin(curve)+1)
plt.xlabel('Trees');plt.ylabel('Validation RMSE');plt.legend();savefig('c07_boosting_rounds')
cutpoints=np.quantile(bxtr[:,0],[.25,.5,.75]);bins=np.searchsorted(cutpoints,bxtr[:,0],side='right')
assert set(bins)<=set(range(4));print(pd.DataFrame({'x':bxtr[:8,0],'bin':bins[:8]}))
print('Stack first-layer feature matrix:',stack.transform(Xtr[:5]).shape)
print('Stack output shape:',stack.predict_proba(Xte[:5]).shape)

## 練習與參考答案

- 硬投票與軟投票何時不同？**當被少數模型的過度自信機率拉動平均時；需檢查校準。**
- 「重複同一模型、同一種子 100 次」能形成有效多樣性嗎？**不能，預測完全相同。**
- AdaBoost 一直放大同一個錯誤標籤會怎樣？**標籤雜訊被持續放大，可能過擬合雜訊。**
- stacking 訓練特徵看起來完美，代表模型好嗎？**否，很可能是用了記住答案的 in-sample 預測。**